# AlexNet: ImageNet Classification with Deep Convolutional Neural Networks (Krizhevsky et al., NeurIPS 2012)
### Triển Khai Bằng PyTorch & Thử Nghiệm Đối Chiếu Giá Trị Học Thuật

> **Tác giả:** Alex Krizhevsky, Ilya Sutskever, Geoffrey E. Hinton  
> **Hội nghị:** NeurIPS / NIPS 2012 (Lake Tahoe, Nevada)  
> **Đóng góp then chốt:** Cú hích lịch sử khai sinh cuộc cách mạng Deep Learning toàn cầu với sự hội tụ của dữ liệu lớn (ImageNet), phần cứng GPU (NVIDIA CUDA), và thuật toán đột phá (ReLU, Dropout, LRN).

---

## 🎯 Mục Tiêu Bài Thực Hành
1. **Kiến trúc AlexNet 8 tầng:** Conv1-5 kết hợp `LocalResponseNorm` (LRN), `Overlapping MaxPool` và FC6-8 với `Dropout`.
2. **Tái hiện Figure 3:** Trích xuất và vẽ ma trận 96 bộ lọc tích chập tầng Conv1 ($11 \times 11 \times 3$) thể hiện các bộ dò cạnh và họa tiết màu sắc.
3. **Academic Value Benchmark:** Đối chiếu định lượng 3 mô hình học sâu:
   - *Mô hình 1 (Kỷ nguyên Tiền-AlexNet):* Mạng dùng hàm kích hoạt Tanh bão hòa, không Dropout $\rightarrow$ Triệt tiêu gradient.
   - *Mô hình 2 (AlexNet Không Điều Hòa):* Dùng ReLU nhưng TẮT Dropout $\rightarrow$ Quá khớp (Overfitting) nghiêm trọng.
   - *Mô hình 3 (AlexNet Hoàn Chỉnh):* ReLU + LRN + Dropout ($p=0.5$) + SGD Momentum $\rightarrow$ Hội tụ nhanh gấp 6 lần, tổng quát hóa cao.
4. **Trực quan hóa đa bảng:** Xuất biểu đồ 4 bảng đối chiếu toàn diện.

In [ ]:
import os
import time
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import matplotlib.pyplot as plt

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"[✓] Thiết bị tính toán được chọn: {device}")

## PHẦN 1: ĐỊNH NGHĨA KIẾN TRÚC MÔ HÌNH ALEXNET BẰNG PYTORCH

AlexNet có 8 tầng học được (learnable layers):
- **5 tầng Convolutional:** Trích xuất đặc trưng phân cấp từ biên cạnh cục bộ đến vật thể hoàn chỉnh.
- **3 tầng Fully Connected:** Phân loại đa lớp xác suất qua Softmax.
- **LocalResponseNorm (LRN):** Chuẩn hóa phản hồi trên $n=5$ kênh lân cận.
- **Dropout ($p=0.5$):** Ngắt ngẫu nhiên nơ-ron ở FC6 và FC7.

In [ ]:
class AlexNet(nn.Module):
    def __init__(self, num_classes=10, use_relu=True, use_dropout=True, use_lrn=True, in_channels=3):
        super(AlexNet, self).__init__()
        self.use_relu = use_relu
        self.use_dropout = use_dropout
        self.use_lrn = use_lrn

        act_layer = nn.ReLU if use_relu else nn.Tanh
        p_drop = 0.5 if use_dropout else 0.0

        norm1 = nn.LocalResponseNorm(size=5, alpha=1e-4, beta=0.75, k=2.0) if use_lrn else nn.Identity()
        norm2 = nn.LocalResponseNorm(size=5, alpha=1e-4, beta=0.75, k=2.0) if use_lrn else nn.Identity()

        self.features = nn.Sequential(
            nn.Conv2d(in_channels, 96, kernel_size=11, stride=4, padding=2),
            act_layer(),
            norm1,
            nn.MaxPool2d(kernel_size=3, stride=2),

            nn.Conv2d(96, 256, kernel_size=5, padding=2),
            act_layer(),
            norm2,
            nn.MaxPool2d(kernel_size=3, stride=2),

            nn.Conv2d(256, 384, kernel_size=3, padding=1),
            act_layer(),

            nn.Conv2d(384, 384, kernel_size=3, padding=1),
            act_layer(),

            nn.Conv2d(384, 256, kernel_size=3, padding=1),
            act_layer(),
            nn.MaxPool2d(kernel_size=3, stride=2),
            nn.AdaptiveAvgPool2d((6, 6))
        )

        self.classifier = nn.Sequential(
            nn.Dropout(p=p_drop),
            nn.Linear(256 * 6 * 6, 1024),
            act_layer(),
            nn.Dropout(p=p_drop),
            nn.Linear(1024, 1024),
            act_layer(),
            nn.Linear(1024, num_classes)
        )

    def forward(self, x):
        feat = self.features(x)
        feat_flat = torch.flatten(feat, 1)
        return self.classifier(feat_flat)

print("[✓] Đã khởi tạo kiến trúc AlexNet PyTorch thành công.")

## PHẦN 2: BỘ NẠP DỮ LIỆU & BỘ SINH DỮ LIỆU TỔNG HỢP

In [ ]:
from alexnet_pytorch import load_data_with_fallback

train_loader, test_loader, num_classes = load_data_with_fallback(n_samples=500, img_size=64, num_classes=8)
print(f"[*] Số lượng mẫu tập huấn luyện: {len(train_loader.dataset)}")
print(f"[*] Số lượng mẫu tập kiểm tra:   {len(test_loader.dataset)}")

## PHẦN 3: BỘ ĐO LƯỜNG CHUẨN MỰC (TOP-1 & TOP-5 ACCURACY)

In [ ]:
from alexnet_pytorch import compute_accuracy_metrics
print("[✓] Đã sẵn sàng hàm compute_accuracy_metrics.")

## PHẦN 4: ĐỐI CHIẾU GIÁ TRỊ HỌC THUẬT (ACADEMIC VALUE BENCHMARK)

Chạy thử nghiệm so sánh 3 trường phái:
1. **Mô Hình 1: Tiền-AlexNet (Tanh, No Dropout)**
2. **Mô Hình 2: AlexNet Không Điều Hòa (ReLU, No Dropout - Quá khớp)**
3. **Mô Hình 3: Full AlexNet (ReLU + LRN + Dropout + SGD Momentum)**

In [ ]:
from alexnet_pytorch import run_academic_value_benchmark

# Chạy đối chuẩn giá trị học thuật
bench_results = run_academic_value_benchmark(device)

## PHẦN 5: TRỰC QUAN HÓA 96 BỘ LỌC CONV1 (TÁI HIỆN FIGURE 3)

Trực quan hóa 96 bộ lọc của tầng Conv1 kích thước $11 \times 11 \times 3$. Chúng ta sẽ quan sát thấy 2 nhóm bộ lọc tự động hình thành:
- Các bộ phát hiện cạnh định hướng Gabor (đơn sắc, tần số cao).
- Các bộ phát hiện đốm màu (color blobs) và gradient diện rộng.

In [ ]:
from alexnet_pytorch import plot_conv1_filters

out_dir = "outputs"
plot_conv1_filters(bench_results["model_alex"], output_dir=out_dir)

# Hiển thị ảnh bộ lọc
filter_img_path = os.path.join(out_dir, "alexnet_filters.png")
if os.path.exists(filter_img_path):
    img_filt = plt.imread(filter_img_path)
    plt.figure(figsize=(14, 9))
    plt.imshow(img_filt)
    plt.axis('off')
    plt.title("Tái Hiện Figure 3: 96 Bộ Lọc Tầng Conv1 Của AlexNet", fontsize=14, fontweight='bold')
    plt.show()

## PHẦN 6: BIỂU ĐỒ ĐỐI CHIẾU HỌC THUẬT CHUYÊN SÂU

Xuất biểu đồ 4 bảng phân tích:
1. **Tốc độ hội tụ mất mát:** Minh chứng ReLU hội tụ nhanh gấp 6 lần so với Tanh.
2. **Khoảng cách Quá Khớp:** Minh chứng Dropout triệt tiêu hiện tượng học vẹt.
3. **Top-1 vs Top-5 Accuracy:** So sánh năng lực phân loại tổng quát trên tập kiểm tra.
4. **Cường độ Gradient Norm:** Minh chứng ReLU chống triệt tiêu gradient ở các tầng sâu.

In [ ]:
from alexnet_pytorch import plot_comprehensive_academic_comparison

plot_comprehensive_academic_comparison(bench_results, output_dir=out_dir)

# Hiển thị biểu đồ đối chiếu
comp_img_path = os.path.join(out_dir, "alexnet_academic_comparison.png")
if os.path.exists(comp_img_path):
    img_comp = plt.imread(comp_img_path)
    plt.figure(figsize=(16, 12))
    plt.imshow(img_comp)
    plt.axis('off')
    plt.title("Kết Quả Đối Chiếu Học Thuật Toàn Diện (AlexNet NeurIPS 2012)", fontsize=14, fontweight='bold')
    plt.show()

## 🎓 KẾT LUẬN & Ý NGHĨA LỊCH SỬ CỦA ALEXNET (NEURIPS 2012)

1. **Khai sinh cuộc cách mạng Deep Learning:** AlexNet đã chứng minh rằng mạng nơ-ron tích chập sâu kết hợp với tính toán GPU song song có thể nghiền nát mọi thuật toán trích xuất đặc trưng thủ công (SIFT/HOG + SVM).
2. **ReLU là bước ngoặt:** Việc chuyển sang ReLU đã cứu mạng nơ-ron sâu khỏi bẫy triệt tiêu gradient, tăng tốc độ huấn luyện gấp nhiều lần.
3. **Dropout là chìa khóa điều hòa:** Cho phép các mô hình hàng chục triệu tham số huấn luyện ổn định mà không bị quá khớp trên dữ liệu phức tạp.
4. **Nền móng cho tương lai:** VGGNet, ResNet, ConvNeXt, và thậm chí cả Vision Transformer đều thừa hưởng trực tiếp triết lý và kiến trúc phân tầng từ công trình vĩ đại này.